In [ ]:
# Import libraries for model training and evaluation.
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

# Build a safe path to the dataset file relative to the notebook location.
file_path = Path('..') / 'data' / 'PGCB_date_power_demand.xlsx'
if not file_path.exists():
    raise FileNotFoundError(f'Dataset file not found: {file_path.resolve()}')

# Read the raw Excel file.
df = pd.read_excel(file_path, sheet_name=0, engine='openpyxl')

# Find the datetime column automatically.
datetime_columns = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
if len(datetime_columns) == 0:
    raise ValueError('No datetime-like column found. Please check the dataset columns.')
datetime_col = datetime_columns[0]
print('Using datetime column:', datetime_col)

# Convert the datetime column and sort chronologically.
df[datetime_col] = pd.to_datetime(df[datetime_col])
df = df.sort_values(datetime_col).reset_index(drop=True)

# Forward-fill missing target values.
df['demand_mw'] = df['demand_mw'].ffill()

# Create simple time features.
df['hour'] = df[datetime_col].dt.hour
df['day'] = df[datetime_col].dt.day
df['month'] = df[datetime_col].dt.month

# Scale the target variable for the models.
scaler = MinMaxScaler()
df['demand_scaled'] = scaler.fit_transform(df[['demand_mw']])

# Choose the features for model training.
feature_columns = ['hour', 'day', 'month']
X = df[feature_columns]
y = df['demand_scaled']

# Split the data chronologically (no shuffling).
split_index = int(len(df) * 0.8)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Train a baseline RandomForestRegressor.
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Train a more advanced XGBoost model.
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

# Predict on the test set.
y_pred_rf = rf_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)

# Inverse transform predictions back to original units.
y_test_inv = scaler.inverse_transform(y_test.values.reshape(-1, 1)).flatten()
y_pred_rf_inv = scaler.inverse_transform(y_pred_rf.reshape(-1, 1)).flatten()
y_pred_xgb_inv = scaler.inverse_transform(y_pred_xgb.reshape(-1, 1)).flatten()

# Calculate and print evaluation metrics.
rf_mae = mean_absolute_error(y_test_inv, y_pred_rf_inv)
rf_rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_rf_inv))
xgb_mae = mean_absolute_error(y_test_inv, y_pred_xgb_inv)
xgb_rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_xgb_inv))

print('Random Forest MAE:', round(rf_mae, 2))
print('Random Forest RMSE:', round(rf_rmse, 2))
print('\nXGBoost MAE:', round(xgb_mae, 2))
print('XGBoost RMSE:', round(xgb_rmse, 2))